# 21 — CNN: CR_div + SG1 微分 (PP_C fix)

nb20 診断: CR_div 値は [0,1] で分散が小さく CNN が学習できなかった (train_rmse=53%)。
Fix: CR_div -> SG1(41,3,1) 微分を追加して分散を増幅。

| 前処理 | nb20 train_rmse | 期待 |
|---|---|---|
| PP_A SNV+SG1 | 12.51% | 正常 |
| PP_C CR_div only | 53.29% | 退化 |
| PP_C2 CR_div+SG1 | ? | Fix 後 |

仮説: 微分で CR_div の局所変化を強調 → CNN が学習可能になる。
成功条件: train_rmse < 20%、OOF 相関 PP_A vs PP_C2 < 0.90

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import load_data, parse_spectra, get_groups, make_submission

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
CLIP_T = 200.0

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta,  _,   X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

print(f'Device: {DEVICE}')
print(f'wn range: {wn.min():.1f} - {wn.max():.1f}')

In [ ]:
class ImprovedCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.block12 = nn.Sequential(
            nn.Conv1d(1,  8,  kernel_size=15, padding=7), nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8,  16, kernel_size=9,  padding=4), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.conv3     = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32))
        self.shortcut3 = nn.Conv1d(16, 32, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool1d(16)
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(32 * 16, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.block12(x.unsqueeze(1))
        h = torch.relu(self.conv3(h) + self.shortcut3(h))
        h = self.pool(h)
        return self.fc(h.view(x.size(0), -1)).squeeze(1)

# ===== 前処理 =====
def pp_snv_sg1(Xtr, Xva):
    def _apply(X):
        A = X.astype(float)
        A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
        return savgol_filter(A, 41, 3, deriv=1, axis=1).astype(np.float32)
    return _apply(Xtr), _apply(Xva)

def _upper_hull_interp(wn_s, R_s):
    hull = []
    for i in range(len(wn_s)):
        while len(hull) >= 2:
            o, a = hull[-2], hull[-1]
            cross = ((wn_s[a]-wn_s[o])*(R_s[i]-R_s[o])
                     - (R_s[a]-R_s[o])*(wn_s[i]-wn_s[o]))
            if cross >= 0: hull.pop()
            else: break
        hull.append(i)
    return np.interp(wn_s, wn_s[hull], R_s[hull])

def pp_crdiv_sg1(Xtr, Xva):
    """CR_div -> SG1(41,3,1): 微分で分散増幅"""
    wn_arr = wn.astype(float)
    def _crdiv(X):
        out = np.zeros_like(X, dtype=float)
        for i, R in enumerate(X.astype(float)):
            h = _upper_hull_interp(wn_arr, R)
            out[i] = R / np.maximum(h, 1e-8)
        return out
    Xtr_cr = _crdiv(Xtr)
    Xva_cr = _crdiv(Xva)
    Xtr_sg = savgol_filter(Xtr_cr, 41, 3, deriv=1, axis=1)
    Xva_sg = savgol_filter(Xva_cr, 41, 3, deriv=1, axis=1)
    return Xtr_sg.astype(np.float32), Xva_sg.astype(np.float32)

# 前処理後の統計を確認
Xtr_a, _ = pp_snv_sg1(X_raw, X_raw[:1])
Xtr_c, _ = pp_crdiv_sg1(X_raw, X_raw[:1])
print(f'PP_A SNV+SG1  : std={Xtr_a.std():.4f}  range=[{Xtr_a.min():.4f}, {Xtr_a.max():.4f}]')
print(f'PP_C2 CRdiv+SG1: std={Xtr_c.std():.4f}  range=[{Xtr_c.min():.4f}, {Xtr_c.max():.4f}]')
print(f'(分散が十分あれば CNN が学習可能)')

PP_CONFIGS = {
    'PP_A_snv_sg1':    pp_snv_sg1,
    'PP_C2_crdiv_sg1': pp_crdiv_sg1,
}

def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan
def rmse_all(yt, yp): return float(np.sqrt(np.mean((yt-yp)**2)))

print('Ready. Params:', sum(p.numel() for p in ImprovedCNN1D().parameters()))

In [ ]:
def train_one(Xtr_pp, ytr, Xva_pp, yva, seed=42,
              n_epochs=100, batch=32, lr=1e-3, patience=20):
    torch.manual_seed(seed)
    np.random.seed(seed)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr_pp).astype(np.float32)
    Xva_s = sc.transform(Xva_pp).astype(np.float32)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch, shuffle=True)
    model = ImprovedCNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit  = nn.HuberLoss(delta=10.0)
    best_val, best_state, best_preds = float('inf'), None, None
    no_improve = 0; stop_ep = n_epochs
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = model(Xva_t)
            vl = crit(vp, yva_t).item()
        if vl < best_val:
            best_val = vl; best_state = copy.deepcopy(model.state_dict())
            best_preds = vp.cpu().numpy(); no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            stop_ep = epoch + 1; break
    model.load_state_dict(best_state)
    return model, sc, best_preds, stop_ep

## GroupKFold CV — PP_A vs PP_C2 (seed=42)

In [ ]:
pp_names   = list(PP_CONFIGS.keys())
pp_fold_r  = {p: [] for p in pp_names}
pp_oof     = {p: [] for p in pp_names}
stop_eps   = {p: [] for p in pp_names}
oof_y_list = []

for fi, (tr, va) in enumerate(SPLITS):
    Xtr_r, Xva_r = X_raw[tr], X_raw[va]
    ytr, yva     = y[tr],     y[va]
    oof_y_list.append(yva)
    fold_preds = []

    for pname, pp_fn in PP_CONFIGS.items():
        Xtr_pp, Xva_pp = pp_fn(Xtr_r, Xva_r)
        _, _, pred, stop_ep = train_one(Xtr_pp, ytr, Xva_pp, yva, seed=SEED)
        r = rmse_le(yva, pred)
        pp_fold_r[pname].append(round(r, 2))
        pp_oof[pname].append(pred)
        stop_eps[pname].append(stop_ep)
        fold_preds.append(pred)
        print(f'  Fold{fi+1} {pname}: {r:.2f}%  stop_ep={stop_ep}')

    avg_pred = np.mean(fold_preds, axis=0)
    print(f'  Fold{fi+1} COMBINED: {rmse_le(yva, avg_pred):.2f}%')
    print()

oof_y = np.concatenate(oof_y_list)

print('=== CV Summary ===')
print(f'{"前処理":<22} F1     F2     F3     F4     F5   Mean')
for p in pp_names:
    fs = pp_fold_r[p]
    print(f'{p:<22} {fs[0]:5.2f}  {fs[1]:5.2f}  {fs[2]:5.2f}  {fs[3]:5.2f}  {fs[4]:5.2f}  {np.mean(fs):.2f}%')

oof_p_comb = np.mean([np.concatenate(pp_oof[p]) for p in pp_names], axis=0)
comb_le = rmse_le(oof_y, oof_p_comb)
print(f'{"COMBINED":<22}                                   {comb_le:.2f}% (overall OOF)')
print()
print('nb18 reference (SNV+SG1 x3seed): CV=21.96%  LB=17.73')
print()

# OOF distribution for each PP
mask170 = oof_y <= 170
for p in pp_names:
    pp_pred = np.concatenate(pp_oof[p])
    print(f'{p}: OOF mean={pp_pred[mask170].mean():.1f}  '
          f'min={pp_pred.min():.1f}  max={pp_pred.max():.1f}')
print(f'COMBINED: mean={oof_p_comb[mask170].mean():.1f}  '
      f'min={oof_p_comb.min():.1f}  max={oof_p_comb.max():.1f}')
print()

# OOF pairwise correlation
pp_all = {p: np.concatenate(pp_oof[p]) for p in pp_names}
r_oof  = np.corrcoef(pp_all['PP_A_snv_sg1'][mask170],
                     pp_all['PP_C2_crdiv_sg1'][mask170])[0,1]
print(f'OOF 相関 PP_A vs PP_C2 (y<=170): r={r_oof:.4f}')
print(f'(nb20 PP_A vs PP_C[no deriv]: r=0.2577)')
print(f'(nb18 ET vs CNN:               r=0.9902)')

In [ ]:
import os; os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
x = np.arange(5); colors = ['steelblue', 'tomato']
ref_nb20 = [34.01, 42.78, 38.24, 41.51, 34.35]  # CR_div no-deriv
for i, (p, col) in enumerate(zip(pp_names, colors)):
    ax.bar(x + (i-0.5)*0.3, pp_fold_r[p], 0.3, label=p.split('_',1)[1], color=col, alpha=0.8)
ax.plot(x, ref_nb20, 'k--', marker='x', ms=6, lw=1, label='CR_div only (nb20)')
ax.set_xticks(x); ax.set_xticklabels([f'F{i+1}' for i in range(5)])
ax.set_ylabel('RMSE_le170 (%)')
ax.set_title('PP_A vs PP_C2 (CR_div+SG1)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
m = oof_y <= 170
ax.scatter(oof_y[m], oof_p_comb[m], s=4, alpha=0.4, label='Combined')
lim = max(oof_y[m].max(), oof_p_comb[m].max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=0.8)
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title(f'OOF combined (r={r_oof:.3f}, RMSE_le170={comb_le:.2f}%)')
ax.grid(True, alpha=0.3)

ax = axes[2]
pa = pp_all['PP_A_snv_sg1'][m]
pc = pp_all['PP_C2_crdiv_sg1'][m]
ax.scatter(pa, pc, s=4, alpha=0.3)
lim2 = max(pa.max(), pc.max()) * 1.05
ax.plot([0, lim2], [0, lim2], 'r--', lw=0.8)
ax.set_xlabel('PP_A pred'); ax.set_ylabel('PP_C2 pred')
ax.set_title(f'PP_A vs PP_C2 scatter (r={r_oof:.4f})')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/nb21_crdiv_sg1_cv.png', dpi=110)
plt.close()
print('Saved: results/nb21_crdiv_sg1_cv.png')

## Full Train → Test Prediction

In [ ]:
print('=== Test predictions ===')

te_preds = {}
for pname, pp_fn in PP_CONFIGS.items():
    avg_stop = int(np.mean(stop_eps[pname]))
    print(f'{pname}: avg_stop={avg_stop}')

    Xtr_pp, Xte_pp = pp_fn(X_raw, X_test_raw)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr_pp).astype(np.float32)
    Xte_s = sc.transform(Xte_pp).astype(np.float32)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(y.astype(np.float32)).to(DEVICE)

    torch.manual_seed(SEED); np.random.seed(SEED)
    model_f  = ImprovedCNN1D().to(DEVICE)
    opt_f    = torch.optim.Adam(model_f.parameters(), lr=1e-3)
    sched_f  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
    crit_f   = nn.HuberLoss(delta=10.0)
    loader_f = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)

    for ep in range(avg_stop):
        model_f.train()
        for xb, yb in loader_f:
            loss = crit_f(model_f(xb), yb)
            opt_f.zero_grad(); loss.backward(); opt_f.step()
        sched_f.step()

    model_f.eval()
    with torch.no_grad():
        te_p = model_f(torch.from_numpy(Xte_s).to(DEVICE)).cpu().numpy()
        tr_rmse = torch.sqrt(torch.mean((model_f(Xtr_t)-ytr_t)**2)).item()
    te_preds[pname] = te_p
    print(f'  train_rmse={tr_rmse:.2f}  test: min={te_p.min():.1f}  '
          f'mean={te_p.mean():.1f}  max={te_p.max():.1f}  >170: {(te_p>170).sum()}')

te_comb = np.clip(np.mean(list(te_preds.values()), axis=0), 0, CLIP_T)
r_te = np.corrcoef(te_preds['PP_A_snv_sg1'], te_preds['PP_C2_crdiv_sg1'])[0,1]
print()
print(f'COMBINED: min={te_comb.min():.1f}  mean={te_comb.mean():.1f}  '
      f'max={te_comb.max():.1f}  >170: {(te_comb>170).sum()}')
print(f'テスト相関 PP_A vs PP_C2: r={r_te:.4f}')

In [ ]:
import os; os.makedirs('../submissions', exist_ok=True)

make_submission(test_meta, te_comb, '../submissions/sub_cnn_crdivsg1.csv')
print('Saved: submissions/sub_cnn_crdivsg1.csv')

print()
print('=== Summary ===')
print(f'{"前処理":<24} {"Folds":42} Mean  train_rmse')
for p in pp_names:
    fs = pp_fold_r[p]
    tp = te_preds[p]
    print(f'{p:<24} {str(fs):42} {np.mean(fs):.2f}%')
print(f'{"COMBINED":<24} {"":42} {comb_le:.2f}%  test_mean={te_comb.mean():.1f}')
print()
print('判断基準:')
print('  OK なら: PP_C2 train_rmse < 20%, OOF 相関 < 0.90, test_mean が現実的')
print(f'  結果:    PP_C2 OOF r={r_oof:.4f}, テスト r={r_te:.4f}, test_mean={te_preds["PP_C2_crdiv_sg1"].mean():.1f}%')
print()
ok = (r_oof < 0.90 and te_preds['PP_C2_crdiv_sg1'].mean() > 25)
print('→ 提出推奨:' if ok else '→ 要確認:', 'sub_cnn_crdivsg1.csv')
print()
print('Reference: nb18 SNV+SG1 x3seed LB=17.73, test_mean=44.8%')